## Extension templates

- Fill-in boilerplate for extending a package
- Shows the class contract
   - methods,
   - properties, etc
- One blueprint per type of feature
- New feature follows the same contract, so it plugs in
- No need to read the codebase first

## A base class

In [1]:
import numpy as np


class BaseModel:
    def fit(self, y):
        y = np.asarray(y, dtype=float)
        if y.ndim != 1:
            raise ValueError("y must be 1D")
        self._fit(y)
        self.is_fitted_ = True
        return self

    def predict(self, steps):
        if not getattr(self, "is_fitted_", False):
            raise RuntimeError("call fit first")
        return self._predict(steps)

    def _fit(self, y):
        raise NotImplementedError

    def _predict(self, steps):
        raise NotImplementedError

## The template

```python
class MyModel(BaseModel):
    # todo: hyper-parameters
    def __init__(self, ...): ...

    # todo: learn from y, store results in attributes ending in _
    def _fit(self, y): ...

    # todo: return `steps` future values
    def _predict(self, steps): ...
```

## Fill it in

In [2]:
class LastValue(BaseModel):
    def _fit(self, y):
        self.last_ = y[-1]

    def _predict(self, steps):
        return np.full(steps, self.last_)


class WindowMean(BaseModel):
    def __init__(self, window=3):
        self.window = window

    def _fit(self, y):
        self.mean_ = y[-self.window :].mean()

    def _predict(self, steps):
        return np.full(steps, self.mean_)

## Use them

In [3]:
y = [1, 2, 3, 4, 5]
print(LastValue().fit(y).predict(3))
print(WindowMean(window=2).fit(y).predict(3))

[5. 5. 5.]
[4.5 4.5 4.5]


## Why it helps

- New model has just two short methods
- Input checks and validation come from the base class

## sktime extension templates

- Fill-in files in `extension_templates/`, one per estimator type
- Same idea: `fit` lives in `BaseForecaster`, you write `_fit`
- Extend sktime in your own code, no fork needed
- Agents use them too: marked gaps, one interface to follow
- Smallest forecaster template: `forecasting_supersimple.py`

```python
from sktime.forecasting.base import BaseForecaster


class MyForecaster(BaseForecaster):
    """Custom forecaster. todo: write docstring."""

    # todo: fill in the capability:multivariate tag
    _tags = {
        "capability:multivariate": False,
        "authors": ["author1", "author2"],
        "maintainers": ["maintainer1", "maintainer2"],
        # do not change these:
        "y_inner_mtype": "pd.DataFrame",
        "X_inner_mtype": "pd.DataFrame",
        "capability:exogenous": True,
        "requires-fh-in-fit": True,
    }

    # todo: add any hyper-parameters and components to constructor
    def __init__(self, parama, paramb="default", paramc=None):
        self.parama = parama
        self.paramb = paramb
        self.paramc = paramc
        super().__init__()

    # todo: implement this, mandatory
    def _fit(self, y, X=None, fh=None):
        # self.fitted_model_param_ = sthsth
        return self

    # todo: implement this, mandatory
    def _predict(self, fh, X=None):
        # row_idx = fh.to_absolute_index(self.cutoff)
        # col_idx = self._get_varnames()
        # y_pred = pd.DataFrame(values, index=row_idx, columns=col_idx)
        ...

    @classmethod
    def get_test_params(cls, parameter_set="default"):
        # todo: set the testing parameters for the estimators
        # params = [{"parama": value1}, {"parama": value2}]
        ...
```

## Tags

- Many more tags exist: 46 for forecasters
- Each one documented in `sktime/registry/_tags.py`
- List them: `all_tags("forecaster", as_dataframe=True)`

## The parts

- `_tags`: what the forecaster supports. Only `capability:multivariate` to choose
- `__init__`: hyper-parameters, stored unchanged
- `_fit`: learn from `y`, save results in attributes ending in `_`
- `_predict`: return a `pd.DataFrame` for the horizon `fh`
- `get_test_params`: settings `check_estimator` uses to test it

## Need more?

- `forecasting_simple.py`: choose input types, use of `X`, when `fh` is needed
- `forecasting.py`: adds `update`, intervals, quantiles, distributions
- Same `check_estimator` for all three

## Up next

- How agents use these templates to implement a model